# 01 - Data Understanding

## Credit Risk Modelling Using Logistic Regression

This notebook profiles the supplied borrower-level data, its quality, and the target prevalence. It reports unusual values but does not change the data.

The outcome SeriousDlqin2yrs is defined as

$$
Y_i =
\begin{cases}
1 & \text{serious delinquency within two years}, \\
0 & \text{otherwise}.
\end{cases}
$$

The intended modelling quantity is Probability of Default (PD), $P(Y_i=1 \mid X_i)$. Here we first estimate the sample prevalence $P(Y=1)$ and $P(Y=0)$.

## 1. Dataset dimensions and variable names

Rows are observations and columns are variables. The supplied file contains an index-like field named `Unnamed: 0`; its role is assessed below, but it is retained in this Stage 1 inspection.

In [11]:
import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = "../data/raw/cs-training.csv"
TARGET = "SeriousDlqin2yrs"

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Observations: {df.shape[0]:,} | Variables: {df.shape[1]:,}")


Loaded: ../data/raw/cs-training.csv
Observations: 150,000 | Variables: 12


In [12]:
print("Shape (observations, variables):", df.shape)
print("Column names:")
for position, name in enumerate(df.columns, start=1):
    print(f"{position:>2}. {name}")

Shape (observations, variables): (150000, 12)
Column names:
 1. Unnamed: 0
 2. SeriousDlqin2yrs
 3. RevolvingUtilizationOfUnsecuredLines
 4. age
 5. NumberOfTime30-59DaysPastDueNotWorse
 6. DebtRatio
 7. MonthlyIncome
 8. NumberOfOpenCreditLinesAndLoans
 9. NumberOfTimes90DaysLate
10. NumberRealEstateLoansOrLines
11. NumberOfTime60-89DaysPastDueNotWorse
12. NumberOfDependents


## 2. Variable roles

Roles are assigned from the column names and project specification. They describe how each field is used in this analysis, not causal effects. Unnamed: 0 appears to be a record identifier, not a borrower risk characteristic.


In [13]:
role_descriptions = {
    "Unnamed: 0": "Index-like observation identifier; assess uniqueness and exclude from risk interpretation.",
    "SeriousDlqin2yrs": "Binary outcome: serious delinquency within two years (1=yes, 0=no).",
    "RevolvingUtilizationOfUnsecuredLines": "Unsecured revolving credit utilization ratio.",
    "age": "Borrower age in years.",
    "NumberOfTime30-59DaysPastDueNotWorse": "Count of 30–59 day delinquency occurrences.",
    "DebtRatio": "Debt-to-income ratio as represented in the supplied data; inspect scale and extremes.",
    "MonthlyIncome": "Borrower monthly income; missing values are present in the supplied data.",
    "NumberOfOpenCreditLinesAndLoans": "Count of open credit lines and loans.",
    "NumberOfTimes90DaysLate": "Count of 90+ day delinquency occurrences.",
    "NumberRealEstateLoansOrLines": "Count of real-estate loans or lines.",
    "NumberOfTime60-89DaysPastDueNotWorse": "Count of 60–89 day delinquency occurrences.",
    "NumberOfDependents": "Number of dependents; missing values are present in the supplied data."
}
variable_roles = pd.DataFrame({
    "variable": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "role / interpretation": [role_descriptions.get(c, "Role to confirm from the data dictionary.") for c in df.columns],
})
display(variable_roles)

,variable,dtype,role / interpretation
0,Unnamed: 0,int64,Index-like observation identifier; assess uniq...
1,SeriousDlqin2yrs,int64,Binary outcome: serious delinquency within two...
2,RevolvingUtilizationOfUnsecuredLines,float64,Unsecured revolving credit utilization ratio.
3,age,int64,Borrower age in years.
4,NumberOfTime30-59DaysPastDueNotWorse,int64,Count of 30–59 day delinquency occurrences.
5,DebtRatio,float64,Debt-to-income ratio as represented in the sup...
6,MonthlyIncome,float64,Borrower monthly income; missing values are pr...
7,NumberOfOpenCreditLinesAndLoans,int64,Count of open credit lines and loans.
8,NumberOfTimes90DaysLate,int64,Count of 90+ day delinquency occurrences.
9,NumberRealEstateLoansOrLines,int64,Count of real-estate loans or lines.


## 3. Data types and missingness

Missingness is reported as a count and as a share of all observations. No imputation is performed here.

In [14]:
missingness = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100),
    "observed_count": df.notna().sum(),
}).sort_values("missing_count", ascending=False)
display(df.dtypes.rename("dtype").to_frame())
display(missingness.round({"missing_percent": 3}))

,dtype
Unnamed: 0,int64
SeriousDlqin2yrs,int64
RevolvingUtilizationOfUnsecuredLines,float64
age,int64
NumberOfTime30-59DaysPastDueNotWorse,int64
DebtRatio,float64
MonthlyIncome,float64
NumberOfOpenCreditLinesAndLoans,int64
NumberOfTimes90DaysLate,int64
NumberRealEstateLoansOrLines,int64


,missing_count,missing_percent,observed_count
MonthlyIncome,29731,19.821,120269
NumberOfDependents,3924,2.616,146076
Unnamed: 0,0,0.000,150000
SeriousDlqin2yrs,0,0.000,150000
age,0,0.000,150000
RevolvingUtilizationOfUnsecuredLines,0,0.000,150000
DebtRatio,0,0.000,150000
NumberOfTime30-59DaysPastDueNotWorse,0,0.000,150000
NumberOfOpenCreditLinesAndLoans,0,0.000,150000
NumberOfTimes90DaysLate,0,0.000,150000


## 4. Distinct values

The table gives the number of distinct non-missing values and a compact preview when a variable has at most 20 distinct values. High-cardinality variables are summarized by count only to keep the inspection readable.

In [15]:
unique_counts = df.nunique(dropna=True)
unique_summary = pd.DataFrame({"distinct_non_missing": unique_counts, "distinct_including_missing": df.nunique(dropna=False)})
unique_summary["values_if_at_most_20"] = [
    ", ".join(map(str, sorted(df[c].dropna().unique().tolist()))) if unique_counts[c] <= 20 else "(omitted: high cardinality)"
    for c in df.columns
]
display(unique_summary)

,distinct_non_missing,distinct_including_missing,values_if_at_most_20
Unnamed: 0,150000,150000,(omitted: high cardinality)
SeriousDlqin2yrs,2,2,"0, 1"
RevolvingUtilizationOfUnsecuredLines,125728,125728,(omitted: high cardinality)
age,86,86,(omitted: high cardinality)
NumberOfTime30-59DaysPastDueNotWorse,16,16,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, ..."
DebtRatio,114194,114194,(omitted: high cardinality)
MonthlyIncome,13594,13595,(omitted: high cardinality)
NumberOfOpenCreditLinesAndLoans,58,58,(omitted: high cardinality)
NumberOfTimes90DaysLate,19,19,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, ..."
NumberRealEstateLoansOrLines,28,28,(omitted: high cardinality)


## 5. Duplicate observations

Report exact duplicate rows both with and without the index-like column. The second check helps determine whether distinct identifier values mask otherwise repeated borrower records. This is an audit only; no rows are removed.

In [16]:
identifier_candidate = "Unnamed: 0" if "Unnamed: 0" in df.columns else None
full_duplicate_count = int(df.duplicated().sum())
dups_no_id = int(df.drop(columns=[identifier_candidate]).duplicated().sum()) if identifier_candidate else full_duplicate_count
print(f"Exact duplicate rows across all supplied columns: {full_duplicate_count:,}")
if identifier_candidate:
    print(f"Exact duplicate rows after excluding {identifier_candidate}: {dups_no_id:,}")

Exact duplicate rows across all supplied columns: 0
Exact duplicate rows after excluding Unnamed: 0: 609


## 6. Descriptive statistics

Numeric summaries include quartiles and extrema. Extreme values are retained in the data and are investigated as potential data-quality problems or influential observations, not automatically labelled errors.

In [17]:
stats = df.describe(include="all").T
display(stats)

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,150000.0,75000.500000,43301.414527,1.0,37500.750000,75000.500000,112500.250000,150000.0
SeriousDlqin2yrs,150000.0,0.066840,0.249746,0.0,0.000000,0.000000,0.000000,1.0
RevolvingUtilizationOfUnsecuredLines,150000.0,6.048438,249.755371,0.0,0.029867,0.154181,0.559046,50708.0
age,150000.0,52.295207,14.771866,0.0,41.000000,52.000000,63.000000,109.0
NumberOfTime30-59DaysPastDueNotWorse,150000.0,0.421033,4.192781,0.0,0.000000,0.000000,0.000000,98.0
DebtRatio,150000.0,353.005076,2037.818523,0.0,0.175074,0.366508,0.868254,329664.0
MonthlyIncome,120269.0,6670.221237,14384.674215,0.0,3400.000000,5400.000000,8249.000000,3008750.0
NumberOfOpenCreditLinesAndLoans,150000.0,8.452760,5.145951,0.0,5.000000,8.000000,11.000000,58.0
NumberOfTimes90DaysLate,150000.0,0.265973,4.169304,0.0,0.000000,0.000000,0.000000,98.0
NumberRealEstateLoansOrLines,150000.0,1.018240,1.129771,0.0,0.000000,1.000000,2.000000,54.0


## 7. Target distribution and sample probabilities

For the observed sample, empirical probabilities are class proportions:

\[
\widehat{P}(Y=1)=\frac{n_1}{n},\qquad \widehat{P}(Y=0)=\frac{n_0}{n}.
\]

These are unconditional sample proportions, not borrower-specific PD predictions.

In [18]:
if TARGET not in df.columns:
    raise KeyError(f"Expected target column {TARGET!r} is absent.")
target_counts = df[TARGET].value_counts(dropna=False).sort_index()
target_table = pd.DataFrame({
    "count": target_counts,
    "proportion": target_counts / len(df),
    "percent": target_counts / len(df) * 100,
})
display(target_table.round({"proportion": 6, "percent": 3}))

if df[TARGET].isna().any() or not set(df[TARGET].dropna().unique()).issubset({0, 1}):
    print("Target check: values are missing or include values other than 0 and 1; inspect before later modelling.")
else:
    p_default = float((df[TARGET] == 1).mean())
    p_no_default = float((df[TARGET] == 0).mean())
    print(f"P(Y=1), serious delinquency within two years: {p_default:.6f} ({p_default:.3%})")
    print(f"P(Y=0), no serious delinquency within two years: {p_no_default:.6f} ({p_no_default:.3%})")

,count,proportion,percent
SeriousDlqin2yrs,,,
0,139974,0.93316,93.316
1,10026,0.06684,6.684


P(Y=1), serious delinquency within two years: 0.066840 (6.684%)
P(Y=0), no serious delinquency within two years: 0.933160 (93.316%)


## 8. Potential identifier assessment

A column is a potential identifier when its values are unique (or nearly unique), function as record labels, and do not describe borrower credit risk. The following checks are diagnostic. The formal project specification identifies `Unnamed: 0` as an observation/index identifier for removal during Stage 2; it must not be interpreted as a risk feature.

In [19]:
identifier_rows = []
for column in df.columns:
    distinct = int(df[column].nunique(dropna=True))
    identifier_rows.append({
        "variable": column,
        "unique_non_missing": distinct,
        "observations": len(df),
        "unique_share": distinct / len(df) if len(df) else np.nan,
        "name_suggests_identifier": any(token in column.lower() for token in ("id", "index", "unnamed")),
    })
identifier_assessment = pd.DataFrame(identifier_rows).sort_values("unique_share", ascending=False)
display(identifier_assessment.round({"unique_share": 4}))
if identifier_candidate:
    print(f"{identifier_candidate!r} is unique: {df[identifier_candidate].is_unique}; interpret as a record/index field, not a risk characteristic.")

,variable,unique_non_missing,observations,unique_share,name_suggests_identifier
0,Unnamed: 0,150000,150000,1.0000,True
2,RevolvingUtilizationOfUnsecuredLines,125728,150000,0.8382,False
5,DebtRatio,114194,150000,0.7613,False
6,MonthlyIncome,13594,150000,0.0906,False
3,age,86,150000,0.0006,False
7,NumberOfOpenCreditLinesAndLoans,58,150000,0.0004,False
9,NumberRealEstateLoansOrLines,28,150000,0.0002,False
8,NumberOfTimes90DaysLate,19,150000,0.0001,False
4,NumberOfTime30-59DaysPastDueNotWorse,16,150000,0.0001,False
10,NumberOfTime60-89DaysPastDueNotWorse,13,150000,0.0001,False


'Unnamed: 0' is unique: True; interpret as a record/index field, not a risk characteristic.


## 9. Suspicious and potentially impossible values

The checks below surface values that merit review. Domain expectations depend on precise definitions and source conventions: for example, a utilization or debt-ratio field may be recorded above 1, and very large counts may reflect source-data coding conventions. Such observations are reported, not corrected or deleted. `age <= 0` is flagged for review; values above 100 are also surfaced for plausibility review.

In [20]:
# General numeric checks: negative values, zeros, and observed extrema.
numeric = df.select_dtypes(include="number")
quality_summary = pd.DataFrame({
    "negative_count": (numeric < 0).sum(),
    "zero_count": (numeric == 0).sum(),
    "minimum": numeric.min(),
    "maximum": numeric.max(),
})
display(quality_summary)

# Explicit domain-oriented flags; thresholds are investigation prompts, not automatic error rules.
checks = {}
if "age" in df:
    checks["age <= 0"] = df["age"] <= 0
    checks["age > 100 (review plausibility)"] = df["age"] > 100
if "RevolvingUtilizationOfUnsecuredLines" in df:
    checks["utilization > 1 (review scale/extremes)"] = df["RevolvingUtilizationOfUnsecuredLines"] > 1
if "DebtRatio" in df:
    checks["DebtRatio > 1 (review scale/extremes)"] = df["DebtRatio"] > 1
count_columns = [c for c in df.columns if c.startswith("NumberOfTime") or c in ("NumberOfOpenCreditLinesAndLoans", "NumberRealEstateLoansOrLines", "NumberOfDependents")]
for column in count_columns:
    checks[f"{column} > 20 (review extreme count)"] = df[column] > 20

suspicious_counts = pd.DataFrame({"flagged_count": {name: int(mask.sum()) for name, mask in checks.items()}})
suspicious_counts["flagged_percent"] = suspicious_counts["flagged_count"] / len(df) * 100
display(suspicious_counts.round({"flagged_percent": 4}))

# Show representative tail values for review without altering the data.
for column in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "age"] + count_columns:
    if column in df:
        print(f"\nLargest observed values — {column}")
        display(df[column].nlargest(5).to_frame())

,negative_count,zero_count,minimum,maximum
Unnamed: 0,0,0,1.0,150000.0
SeriousDlqin2yrs,0,139974,0.0,1.0
RevolvingUtilizationOfUnsecuredLines,0,10878,0.0,50708.0
age,0,1,0.0,109.0
NumberOfTime30-59DaysPastDueNotWorse,0,126018,0.0,98.0
DebtRatio,0,4113,0.0,329664.0
MonthlyIncome,0,1634,0.0,3008750.0
NumberOfOpenCreditLinesAndLoans,0,1888,0.0,58.0
NumberOfTimes90DaysLate,0,141662,0.0,98.0
NumberRealEstateLoansOrLines,0,56188,0.0,54.0


,flagged_count,flagged_percent
age <= 0,1,0.0007
age > 100 (review plausibility),13,0.0087
utilization > 1 (review scale/extremes),3321,2.2140
DebtRatio > 1 (review scale/extremes),35137,23.4247
NumberOfTime30-59DaysPastDueNotWorse > 20 (review extreme count),269,0.1793
NumberOfOpenCreditLinesAndLoans > 20 (review extreme count),3980,2.6533
NumberOfTimes90DaysLate > 20 (review extreme count),269,0.1793
NumberRealEstateLoansOrLines > 20 (review extreme count),10,0.0067
NumberOfTime60-89DaysPastDueNotWorse > 20 (review extreme count),269,0.1793
NumberOfDependents > 20 (review extreme count),0,0.0000



Largest observed values — RevolvingUtilizationOfUnsecuredLines


,RevolvingUtilizationOfUnsecuredLines
85489,50708.0
31414,29110.0
16956,22198.0
149160,22000.0
149279,20514.0



Largest observed values — DebtRatio


...

,NumberRealEstateLoansOrLines
30587,54
104198,32
65728,29
18259,26
68909,25



Largest observed values — NumberOfTime60-89DaysPastDueNotWorse


,NumberOfTime60-89DaysPastDueNotWorse
1733,98
2286,98
3884,98
4417,98
4705,98



Largest observed values — NumberOfDependents


,NumberOfDependents
6299,20.0
128034,13.0
10618,10.0
12982,10.0
22698,10.0


## 10. Class imbalance and evaluation implications

The target distribution above shows the prevalence of serious delinquency in this sample. With an infrequent positive class, a classifier that labels every borrower as non-delinquent could achieve apparently high accuracy while identifying no delinquent borrowers. Accuracy therefore does not, by itself, describe risk ranking, sensitivity to defaults, or the quality of probability estimates.

Later model validation should emphasize **discrimination** (such as ROC-AUC, Gini, and KS), **calibration** (such as Brier score and observed versus predicted default rates), and risk segmentation. Threshold-based measures such as precision and recall can supplement these, with any decision threshold justified by the intended use. No such model metrics are calculated in this data-understanding notebook.

## Stage 1 summary

- The supplied dataset contains **150,000 observations and 12 variables**.
- `SeriousDlqin2yrs` records serious delinquency within two years. There are **10,026** positive outcomes and **139,974** negative outcomes: \(\widehat{P}(Y=1)=0.06684\) (6.684%) and \(\widehat{P}(Y=0)=0.93316\) (93.316%). The positive class is imbalanced, so accuracy alone would not adequately assess a future PD model.
- Missing values occur in `MonthlyIncome` (**29,731; 19.821%**) and `NumberOfDependents` (**3,924; 2.616%**).
- `Unnamed: 0` is unique and index-like. There are **0** exact duplicate rows with it included and **609** duplicate rows when it is excluded. These records are reported only; no rows are removed here.
- Review flags include **1** observation with `age = 0`, **13** with age above 100, **3,321** utilization values above 1, and **35,137** debt ratios above 1. These are inspection findings; values have not been changed or presumed invalid.
- The delinquency-count variables reach **98**, and other predictors have large observed maxima. Extreme values are surfaced for Stage 2 review, with no automatic removal or transformation in this notebook.
- No cleaning, imputation, feature engineering, or modelling was performed. Those steps remain in their later project stages.
